<a href="https://colab.research.google.com/github/CarlosMendez1997Sei/WETSAT_v2/blob/main/2_Modelling_WETSAT_Google_Colab/User_Version_Kmeans_WetSAT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Google Colaboratory\
GitHub\
Wetlands flooding extent and trends using SATellite data and Machine Learning WETSAT\
Code Developed by\
Carlos Mendez\
Sebastian Palomino\
David Zamora

# **User Version K-means Model **

# Install packages and libraries used in WETSAT


In [27]:
###################################### Artificial Intelligence Frameworks #####################################################
# scikit-learn Framework
!pip install scikit-learn
###################################### Data, Geoprocessing and Graphics libraries #####################################################
!pip install rasterio
!pip install matplotlib
!pip install numpy
!pip install contextily
!pip install pandas

# Import libraries and packages

In [28]:
## AI packages
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
from sklearn.preprocessing import RobustScaler

## Processing packages
import os # Provides functions to interact with the operating system (paths, directories).
import re # Regular expressions for pattern matching filenames.
import random # Random number generation (used for sampling pixels)
import warnings # To manage warning messages.
from glob import glob # Utility to find files matching a pattern in directories.
import matplotlib.pyplot as plt
import joblib          # For saving/loading Python objects (scaler, KMeans model).
import numpy as np     # Numerical computing library (arrays, math).
import pandas as pd    # Data manipulation library (tables, CSV export).
import rasterio        # Library to read/write raster (GeoTIFF) files.
from rasterio.errors import NotGeoreferencedWarning  # Specific warning type to suppress.
warnings.filterwarnings("ignore", category=NotGeoreferencedWarning)
from collections import Counter
from google.colab import files
from google.colab import data_table
import urllib.request

# Create Buttoms and import files

## Create Buttom to upload VH Images

In [29]:
# Define vh_path
vh_path = "/content/WetSAT/VH"

# Create vh path if doesn't exist
os.makedirs(vh_path, exist_ok=True)

# Upload files
uploadvh = files.upload()

# Store files in the VH path
for filename, content in uploadvh.items():
    file_path = os.path.join(vh_path, filename)
    with open(file_path, "wb") as f:
        f.write(content)
    print(f"Files stored in: {file_path}")

Saving 20180101_VH.tif to 20180101_VH (1).tif
Saving 20180113_VH.tif to 20180113_VH (1).tif
Saving 20180125_VH.tif to 20180125_VH (1).tif
Saving 20180206_VH.tif to 20180206_VH (1).tif
Saving 20180218_VH.tif to 20180218_VH (1).tif
Files stored in: /content/WetSAT/VH/20180101_VH (1).tif
Files stored in: /content/WetSAT/VH/20180113_VH (1).tif
Files stored in: /content/WetSAT/VH/20180125_VH (1).tif
Files stored in: /content/WetSAT/VH/20180206_VH (1).tif
Files stored in: /content/WetSAT/VH/20180218_VH (1).tif


## Create Buttom to upload VV Images

In [ ]:
# Define vv_path
vv_path = "/content/WetSAT/VV"

# Create vv path if doesn't exist
os.makedirs(vv_path, exist_ok=True)

# Upload files
uploadvv = files.upload()

# Store files in the VH path
for filename, content in uploadvv.items():
    file_path = os.path.join(vv_path, filename)
    with open(file_path, "wb") as f:
        f.write(content)
    print(f"Files stored in: {file_path}")

Saving 20180101_VV.tif to 20180101_VV.tif
Saving 20180113_VV.tif to 20180113_VV.tif
Saving 20180125_VV.tif to 20180125_VV.tif
Saving 20180206_VV.tif to 20180206_VV.tif
Saving 20180218_VV.tif to 20180218_VV.tif
Files stored in: /content/WetSAT/VV/20180101_VV.tif
Files stored in: /content/WetSAT/VV/20180113_VV.tif
Files stored in: /content/WetSAT/VV/20180125_VV.tif
Files stored in: /content/WetSAT/VV/20180206_VV.tif
Files stored in: /content/WetSAT/VV/20180218_VV.tif


### Calculate the PR index in new AOI

In [ ]:
## Access the VH Path
vh_path = "/content/WetSAT/VH"
## Access the VV Path
vv_path = "/content/WetSAT/VV"
## Create the PR Index path
pr_path = "/content/WetSAT/PR_index"

# Create output directory if it doesn't exist
os.makedirs(pr_path, exist_ok=True)

# List VH and VV files (assuming matching filenames)
vh_files = sorted([f for f in os.listdir(vh_path) if f.endswith(".tif")]) # Search files ending with .tif
vv_files = sorted([f for f in os.listdir(vv_path) if f.endswith(".tif")]) # Search files ending with .tif

# Loop through files and compute PR index scaled between 0 and 1
for vh_file, vv_file in zip(vh_files, vv_files):
    # Read the vh_files and vv_files
    with rasterio.open(os.path.join(vh_path, vh_file)) as vh_src, \
         rasterio.open(os.path.join(vv_path, vv_file)) as vv_src:

        vh = vh_src.read(1).astype("float32") # Converts the data to 32-bit float
        vv = vv_src.read(1).astype("float32") # Converts the data to 32-bit float

        pr_index_diff = vh - vv  # Compute PR index as difference

        # Save PR index difference
        profile = vh_src.profile
        profile.update(dtype="float32", count=1)

        pr_filename = f"PR_diff_{vh_file}"
        with rasterio.open(os.path.join(pr_path, pr_filename), "w", **profile) as dst:
            dst.write(pr_index_diff, 1)

print("PR index (VH - VV) computation complete.")

PR index (VH - VV) computation complete.


# Create Multibands Images (VV,VH and PR)

In [ ]:
# Create Output path
multi_dir = "/content/WetSAT/data_cut_dB"
os.makedirs(multi_dir, exist_ok=True)

# List VH, VV and PR files (assuming matching filenames)
vh_files = sorted([f for f in os.listdir(vh_path) if f.endswith(".tif")])
vv_files = sorted([f for f in os.listdir(vv_path) if f.endswith(".tif")])
pr_files = sorted([f for f in os.listdir(pr_path) if f.endswith(".tif")])

# Create looping to create multiband images
for vh_file, vv_file, pr_file in zip(vh_files, vv_files, pr_files):
    # Open the vv,vh and pr files
    with rasterio.open(os.path.join(vv_path, vv_file)) as vv_src, \
         rasterio.open(os.path.join(vh_path, vh_file)) as vh_src, \
         rasterio.open(os.path.join(pr_path, pr_file)) as pr_src:

        # Converts the data to 32-bit float
        vv = vv_src.read(1).astype("float32")
        vh = vh_src.read(1).astype("float32")
        pr = pr_src.read(1).astype("float32")

        # Create profile output with 3 bands (vv,vh and pr)
        profile = vv_src.profile
        profile.update(dtype="float32", count=3)

        # Create ouput file name based in: YYYY/MM/DD_multi.tif
        # Use example of VV to rename files
        date = re.search(r"(20\d{6})", vv_file).group(1)
        out_name = f"{date}_multi.tif"
        out_path = os.path.join(multi_dir, out_name) # Set path and names

        with rasterio.open(out_path, "w", **profile) as dst:
            dst.write(vv, 1)  # Band 1: VV
            dst.write(vh, 2)  # Band 2: VH
            dst.write(pr, 3)  # Band 3: PR

    print(f"Multiband image (vv,vh and pr) create with the name: {out_name}")

print("\n The process to create stack multibands (VV, VH y PR) finished.")

Multiband image (vv,vh and pr) create with the name: 20180101_multi.tif
Multiband image (vv,vh and pr) create with the name: 20180113_multi.tif
Multiband image (vv,vh and pr) create with the name: 20180125_multi.tif
Multiband image (vv,vh and pr) create with the name: 20180206_multi.tif
Multiband image (vv,vh and pr) create with the name: 20180218_multi.tif

 The process to create stack multibands (VV, VH y PR) finished.


## Create directory when data will be stored

In [ ]:
DATA_DIR = "/content/WetSAT/data_cut_dB"
PATTERN  = re.compile(r"^(\d{8})_multi\.tif$", re.IGNORECASE)

# Import the Kmeans Models and Scalers

In [ ]:
# Create backup from GitHub to import K-means model
# Load K-means model
model_url = "https://raw.githubusercontent.com/sei-latam/WETSAT_v2/main/2_Modelling_WETSAT_Google_Colab/Model_Kmeans.joblib"
urllib.request.urlretrieve(model_url, "/content/WetSAT/data_cut_dB/Model_Kmeans.joblib")
kmeans = joblib.load("/content/WetSAT/data_cut_dB/Model_Kmeans.joblib")

# Load Scalers
scaler_url = "https://raw.githubusercontent.com/sei-latam/WETSAT_v2/main/2_Modelling_WETSAT_Google_Colab/Labels_Scaler_Kmeans.joblib"
urllib.request.urlretrieve(scaler_url, "/content/WetSAT/data_cut_dB/Labels_Scaler_Kmeans.joblib")
scaler = joblib.load("/content/WetSAT/data_cut_dB/Labels_Scaler_Kmeans.joblib")

## Visualize Kmeans model

In [ ]:
kmeans

KMeans(algorithm='auto', n_clusters=5, n_init=10, random_state=42)

## Visualize Clusters and Centroids

In [ ]:
print("Number of clusters:", kmeans.n_clusters)
print("Centroid (in normalized space):")
print(kmeans.cluster_centers_)

Number of clusters: 5
Centroid (in normalized space):
[[  0.25175595   0.2017641    0.12446242]
 [-21.696985   -19.891098    -2.09989   ]
 [ -6.1808124   -5.625261    -0.5906805 ]
 [ -1.3137195   -0.71640384  -0.56257814]
 [-13.543598   -13.184584    -0.4963554 ]]


## Upload Scaler path

In [ ]:
SCALER_PATH = os.path.join(DATA_DIR, "Labels_Scaler_Kmeans.joblib")
MODEL_PATH  = os.path.join(DATA_DIR, "Model_Kmeans.joblib")

OUTPUT_SUFFIX = "_class.tif"   # Labels 1-5
OVERWRITE = True

# Create functions to verify previously classification

In [ ]:
def list_multis(folder):
    # Define a function that lists all raster files ending with "_multi.tif" in a given folder.
    files = sorted(glob(os.path.join(folder, "*_multi.tif")))
    # Use glob to find all files matching the pattern "*_multi.tif" inside the folder.
    # sorted() ensures the list is ordered alphabetically (useful for reproducibility).
    items = []
    # Initialize an empty list to store (date, filepath) tuples.
    for f in files:
        # Iterate through each file found.
        name = os.path.basename(f)
        # Extract just the filename (without directory path).
        m = PATTERN.match(name)
        # Apply the regex pattern defined earlier (PATTERN) to check if filename matches "YYYYMMDD_multi.tif".
        if m:
            # If the filename matches the expected pattern:
            date = m.group(1)
            # Extract the date string (first capturing group in regex, i.e., YYYYMMDD).
            items.append((date, f))
            # Store a tuple (date, full filepath) in the items list.
    return items
    # Return the list of tuples. Each element contains the acquisition date and the file path.

def classify_one(path, scaler, kmeans):
    """
    Clasifica una imagen multibanda (VV,VH,PR), retornando:
    - labels_uint8: raster of cluster labels (uint8)
    - meta_salida: metadata dictionary for saving output raster
    """
    with rasterio.open(path) as src:
        # Open the raster file using rasterio (context manager ensures proper closing).
        if src.count < 3:
            # Check that the raster has at least 3 bands (VV, VH, PR).
            raise ValueError(f"{os.path.basename(path)} no tiene 3 bandas (VV,VH,PR).")
        vv = src.read(1).astype(np.float32)
        vh = src.read(2).astype(np.float32)
        pr = src.read(3).astype(np.float32)
        # Read bands 1, 2, and 3 as float32 arrays:
        # Band 1 = VV, Band 2 = VH, Band 3 = PR.
        nodata = src.nodata  # de VV, por diseño
        # Get the NoData value defined in the raster (usually from band 1).
        if nodata is not None:
            mask = (vv != nodata)
            # If NoData is defined, mask out pixels equal to nodata in VV.
        else:
            mask = np.isfinite(vv) & np.isfinite(vh) & np.isfinite(pr)
            # Otherwise, mask out any non-finite values (NaN, inf) across all bands.
        mask &= np.isfinite(vv) & np.isfinite(vh) & np.isfinite(pr)
        # Ensure mask excludes any non-finite values in all three bands.
        H, W = vv.shape
        # Get raster dimensions (height and width).
        X = np.stack([vv, vh, pr], axis=2).reshape(-1, 3)
        # Stack the three bands into a 3D array (H, W, 3).
        # Then reshape into a 2D array of shape (H*W, 3), where each row = pixel vector [VV, VH, PR].
        valid_idx = mask.reshape(-1)
        # Flatten the mask to match the reshaped pixel array.
        X_valid = X[valid_idx]
        # Extract only valid pixels (those not masked).
        # Escalar y predecir
        X_scaled = scaler.transform(X_valid)
        # Apply the trained scaler (RobustScaler) to normalize valid pixel values.

        labels = kmeans.predict(X_scaled)  # 0..K-1
        # Predict cluster labels using the trained KMeans model.
        # Labels are integers from 0 to K-1.
        # Convertir a 1-5
        labels_1k = (labels + 1).astype(np.uint8)
        # Shift labels to 1..K (instead of 0..K-1) for easier interpretation in GIS.
        # Convert to uint8 (saves memory and is raster-compatible).
        # Reconstruir raster de clases
        out = np.zeros(H * W, dtype=np.uint8)
        # Initialize output raster as a flat array of zeros (background).

        out[:] = 0  # 0 = background si hubiera celdas inválidas (no debería si no hay nodata)
        # Explicitly set all values to 0 (background).
        out[valid_idx] = labels_1k
        # Assign predicted cluster labels to valid pixels.
        out = out.reshape(H, W)
        # Reshape back to raster dimensions (H, W).

        # Meta de salida: igual al multibanda, 1 banda
        meta = src.meta.copy()
        # Copy metadata from source raster.
        meta.update({
            "count": 1,       # Output raster has 1 band (cluster labels).
            "dtype": "uint8", # Data type is unsigned 8-bit integer.
            "nodata": 0       # Background value = 0 (represents NoData).
        })
        for k in ["compress", "tiled", "predictor", "zlevel"]:
            meta.pop(k, None)
            # Remove optional metadata keys that may cause issues when saving output.
        return out, meta
        # Return the classified raster (labels) and the updated metadata.

In [ ]:
def main():
    if not (os.path.exists(SCALER_PATH) and os.path.exists(MODEL_PATH)):
        raise SystemExit(".joblib Files not found")

    scaler = joblib.load(SCALER_PATH)
    kmeans = joblib.load(MODEL_PATH)

    items = list_multis(DATA_DIR)
    if not items:
        raise SystemExit("No se encontraron *_multi.tif en la carpeta de entrada.")

    for date, in_path in items:
        out_path = os.path.join(DATA_DIR, f"{date}{OUTPUT_SUFFIX}")
        if (not OVERWRITE) and os.path.exists(out_path):
            print(f"[SALTA] Ya existe {os.path.basename(out_path)} y OVERWRITE=False")
            continue

        labels_raster, meta = classify_one(in_path, scaler, kmeans)

        with rasterio.open(out_path, "w", **meta) as dst:
            dst.write(labels_raster, 1)

        print(f"[OK] Classification: {os.path.basename(out_path)}")

if __name__ == "__main__":
    main()
    # Standard Python entry point.
    # Ensures that main() runs only when the script is executed directly,
    # not when imported as a module.

[OK] Classification: 20180101_class.tif
[OK] Classification: 20180113_class.tif
[OK] Classification: 20180125_class.tif
[OK] Classification: 20180206_class.tif
[OK] Classification: 20180218_class.tif
